# Megaline Plan Recommendation Model
## Project Overview
Megaline wants to recommend newer plans to customers on grandfathered / legacy plans. This project builds models to predict the right plan based on given subscriber behavior data. This data was cleaned and engineered prior in a different project (Sprint 4)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score


##### We want to import all the necessary libraries and tools we will need to make our models. Because we want to find the highest accuracy, we want to try different models so we will be using each we know of so far, Forests, Linear and Logistic Regression.

In [4]:
df = pd.read_csv('/datasets/users_behavior.csv')
print(df.shape)
print(df.info())
df.head()

(3214, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [5]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

Here, we want to define our features and target. The model will be using the calls, minutes, messages, mb_used as its features and the target would be the plan that they are on, or the plan the model would further recommend.

In [9]:
# First split, 20% for test and 80% for train / validation.
features_train_val, features_test, target_train_val, target_test = train_test_split(features, target, test_size = 0.2, random_state=42)
# Second split, 20% for valaidation and 60% for training, giving us a 60/20/20 split.
features_train, features_valid, target_train, target_valid = train_test_split(features_train_val, target_train_val, test_size = 0.25, random_state=42)

print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)

(1928, 4)
(643, 4)
(643, 4)


Here I decided to use a 60/20/20 split. We want to make sure a majority of our data is spent on the training aspect, so that our model can get good results. Not enough data to train on will skew our results and possibly ruin the entire project. I did try one model with a different split (50/25/25) and got significantly worse performance due to not enough training.

In [31]:

# Now that we have confirmed our data has been split, let's run some models! First, we will set up all three models with the training data to then validate.

treemodel = DecisionTreeClassifier (max_depth = 5, min_samples_split = 10, random_state= 42)
treemodel.fit(features_train,target_train)
predictions_valid = treemodel.predict(features_valid)
accuracy_tree = accuracy_score(target_valid, predictions_valid)
print("Validation accuracy of the Tree Model:", accuracy_tree)

forestmodel = RandomForestClassifier (n_estimators = 200, max_features = 0.2, max_depth = 5, min_samples_split = 10, random_state = 42)
forestmodel.fit(features_train,target_train)
predictions_valid = forestmodel.predict(features_valid)
accuracy_forest = accuracy_score(target_valid,predictions_valid)
print("Validation accuracy of the Forest Model:",accuracy_forest)

logregmodel = LogisticRegression(max_iter = 1000, class_weight = 'balanced', random_state = 42)
logregmodel.fit(features_train,target_train)
predictions_valid = logregmodel.predict(features_valid)
accuracy_logreg = accuracy_score(target_valid,predictions_valid)
print("Validation accuracy of the Logistic Regression Model:", accuracy_logreg)

print(target_train.value_counts(normalize=True))


Validation accuracy of the Tree Model: 0.7713841368584758
Validation accuracy of the Forest Model: 0.8102643856920684
Validation accuracy of the Logistic Regression Model: 0.38880248833592534
0    0.686203
1    0.313797
Name: is_ultra, dtype: float64


## Model Comparison Results

I could have used a gridsearch to fine tune the parameters, but for this project and such a small amount of data, I don't think it would have done us any good as there isn't enough data to have meaningful impacts of fine tuning the parameters.

Our tree model gave us on average a 77.5% accuracy with manually tuning depth from 3 to 10, samples split from 5 to 10. With the threshold being 75%, this model barely gets us by but we will keep it for consideration.

Our forest model had the highest acccuracy out of all three, going as high at 81%. Something to note is that when tuning the n_estimators, 200 was the sweetspot, as both 100 and 300 gave us 80.8%. Diminishing returns going any higher, probably due to the relatively small dataset.

Our logreg model is completely off, most likely due to feature scaling. Because the numbers of calls, data used, etc for each customer can wildly vary, the model doesn't have the right 'picture' to know how to predict future data. Tweaking the iteration numbers did not save the model either. We also then printed value counts to confirm that it was not a class issue, just a scaling issue.

We will be using the forest model to test our data fully!


In [35]:
# we have already built the model above so we just need to re-enter variables!

predictions_test = forestmodel.predict(features_test)
accuracy_test = accuracy_score(target_test, predictions_test)
print("Test accuracy:", accuracy_test)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(target_test,predictions_test))
print(classification_report(target_test,predictions_test))


dummy = DummyClassifier(strategy='most_frequent', random_state = 42)
dummy.fit(features_train,target_train)
predictions_dummy = dummy.predict(features_test)
accuracy_dummy = accuracy_score(target_test,predictions_dummy)
print("Dummy accuracy:",accuracy_dummy)

Test accuracy: 0.8087091757387247
[[434  21]
 [102  86]]
              precision    recall  f1-score   support

           0       0.81      0.95      0.88       455
           1       0.80      0.46      0.58       188

    accuracy                           0.81       643
   macro avg       0.81      0.71      0.73       643
weighted avg       0.81      0.81      0.79       643

Dummy accuracy: 0.7076205287713841


# Forest Model Results

After running the forest model, we can see that our forest has a prediction rate of 80.8%. We have compared this to the dummy model that is accurate 70% of the time, meaning our model is 10.7% better! Fantastic! Sanity check passed.
Additionally, a confusion matrix and classification report were generated to deeper evaluate model performance. The model misses a fair amount of the Ultra subscribers, suggesting class imbalace is still impacting performance despite passing the overall threshold. 
